# Kuliah #06 - Interferensi: Pembuktian Konsep dan Implementasi Matriks

Notebook ini disusun sebagai pendamping modul kuliah **Serial Mekanika Kuantum Minimalis 2.0: Kuliah #06 - Interferensi**. 

Secara struktur, notebook ini dibagi menjadi beberapa bagian utama sesuai dengan sub-bab di dalam diktat:
1. **Interferensi Gelombang Klasik**: Penjabaran matriks Jones untuk susunan interferometer polarisasi (Persamaan 1 – 7), pembuktian aljabar matriks pada setiap tahap perambatan, dan verifikasi modulasi intensitas keluaran $I = \frac{I_i}{2}(1 + \cos\phi)$.
2. **Interferensi Foton**: Analisis perangkat interferometer sebagai penguji perilaku dualitas gelombang-partikel foton tunggal (Gambar 4 & 5), pembuktian mengapa pemblokiran lintasan merusak superposisi, dan fenomena interferensi foton tunggal menempuh kedua lintasan sekaligus.
3. **Latihan (Soal-Jawab)**:
   - **Soal 1**: Pembuktian aljabar bahwa matriks Jones $\mathbf{J} = \begin{pmatrix} 0 & e^{i\phi} \\ 1 & 0 \end{pmatrix}$ bersifat uniter dan melestarikan amplitudo/norma vektor polarisasi masukan.
   - **Soal 2**: Pembuktian matematis konservasi energi pada interferometer lengkap dengan menganalisis kombinasi intensitas saluran $+45^\circ$ dan $-45^\circ$.
   - **Soal 3**: Analisis kuantum amplitudo probabilitas untuk masukan $|V\rangle$ (tidak terjadi interferensi) berbanding masukan $|R\rangle$ (interferensi konstruktif menghasilkan $-i|L\rangle$).

Semua pembuktian matematis dijabarkan menggunakan aljabar matriks dan vektor kolom/baris yang diturunkan langsung dari kalkulus Jones dan aljabar Dirac. Kode Python menggunakan modul `numpy` dan `sympy` disediakan untuk memverifikasi setiap konsep secara numerik maupun simbolik.

In [1]:
import math
import numpy as np
import sympy as sp

np.set_printoptions(precision=4, suppress=True)
sp.init_printing()

def clean_array(A, tol=1e-12):
    """Membersihkan nilai elemen matriks/vektor yang mendekati nol akibat kesalahan pembulatan numerik (floating-point)."""
    A = np.array(A, dtype=complex)
    A[np.abs(A.real) < tol] = 1j * A[np.abs(A.real) < tol].imag
    A[np.abs(A.imag) < tol] = A[np.abs(A.imag) < tol].real
    if np.all(np.abs(A.imag) < tol):
        A = A.real
    return A

def print_matrix(name, M):
    M = clean_array(M)
    print(f"{name} =")
    print(M)
    print()

def print_ket(name, v):
    v = clean_array(v)
    print(f"|{name}> =")
    print(v)
    print()

# Fungsi perkalian dalam <bra|ket> dan norma
def inner_product(bra, ket):
    bra = np.array(bra, dtype=complex).flatten()
    ket = np.array(ket, dtype=complex).flatten()
    return np.vdot(bra, ket)

def norm(v):
    return np.sqrt(np.abs(inner_product(v, v)))

# Operasi Adjoint dan Outer Product
def adjoint(M):
    return np.conjugate(np.transpose(M))

def outer_product(ket, bra):
    ket = np.array(ket, dtype=complex).reshape(-1, 1)
    bra = np.array(bra, dtype=complex).reshape(1, -1)
    return ket @ np.conjugate(bra)

# Vektor Keadaan Dasar dalam Basis Horizontal-Vertikal (HV)
ket_H = np.array([[1], [0]], dtype=complex)
ket_V = np.array([[0], [1]], dtype=complex)

ket_plus45  = (1 / np.sqrt(2)) * np.array([[1], [1]], dtype=complex)
ket_minus45 = (1 / np.sqrt(2)) * np.array([[1], [-1]], dtype=complex)

ket_L = (1 / np.sqrt(2)) * np.array([[1], [1j]], dtype=complex)
ket_R = (1 / np.sqrt(2)) * np.array([[1], [-1j]], dtype=complex)

# Matriks Jones Dasar
J_H = np.array([[1, 0], [0, 0]], dtype=complex)
J_V = np.array([[0, 0], [0, 1]], dtype=complex)

def J_phi(phi):
    """Matriks pergeseran fase phi pada komponen horizontal relatif terhadap vertikal."""
    return np.array([[np.exp(1j * phi), 0],
                     [0, 1]], dtype=complex)

def J_halfwave(theta_deg):
    """Matriks pelat setengah gelombang dengan sumbu cepat bersudut theta terhadap horizontal."""
    theta = np.deg2rad(theta_deg)
    return np.array([[np.cos(2 * theta),  np.sin(2 * theta)],
                     [np.sin(2 * theta), -np.cos(2 * theta)]], dtype=complex)

# Polarisator +45 dan -45 derajat
J_plus45  = outer_product(ket_plus45, ket_plus45)
J_minus45 = outer_product(ket_minus45, ket_minus45)

print("Setup selesai. Keadaan basis dan matriks Jones dasar siap digunakan.")

Setup selesai. Keadaan basis dan matriks Jones dasar siap digunakan.


# 1. Interferensi Gelombang Klasik

## 1.1 Penulisan Ulang Persamaan

Dalam optika klasik, kita meninjau susunan dua polarisator pemisah berkas $PA_{HV}$ seperti pada Gambar 1. Polarisator pertama memisahkan berkas sinar datang terpolarisasi $+45^\circ$ menjadi komponen horizontal ($H$) dan vertikal ($V$). Polarisator kedua menggabungkannya kembali.

1. **Penggabungan Langsung tanpa Pergeseran Fase (Gambar 1)**:
   Untuk salah satu jalur berkas ketika terbagi dua, $PA_{HV}$ pertama berperilaku sebagai polarisator vertikal $\mathbf{J}_V$. Untuk berkas satunya lagi bertindak sebagai polarisator horizontal $\mathbf{J}_H$. $PA_{HV}$ kedua menggabungkan kembali berkas-berkas tersebut sehingga matriks Jones efektif dari dua lintasan adalah:

   $$\mathbf{J} = \mathbf{J}_V + \mathbf{J}_H = \begin{pmatrix} 0 & 0 \\ 0 & 1 \end{pmatrix} + \begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix} = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} = \mathbf{I}, \tag{1}$$

   yang merupakan matriks identitas.

2. **Perbedaan Fase Antara Dua Lintasan**:
   Jika kedua komponen polarisasi memiliki panjang lintasan yang berbeda, timbul pergeseran fase $\phi$ pada polarisasi horizontal relatif terhadap polarisasi vertikal:

   $$\mathbf{J}_\phi = \begin{pmatrix} e^{i\phi} & 0 \\ 0 & 1 \end{pmatrix}. \tag{2}$$

   Matriks Jones efektif interferometer Gambar 1 dengan pergeseran fase menjadi:

   $$\mathbf{J} = \mathbf{J}_V + \mathbf{J}_\phi \mathbf{J}_H = \begin{pmatrix} 0 & 0 \\ 0 & 1 \end{pmatrix} + \begin{pmatrix} e^{i\phi} & 0 \\ 0 & 1 \end{pmatrix} \begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix} = \begin{pmatrix} e^{i\phi} & 0 \\ 0 & 1 \end{pmatrix}. \tag{3}$$

3. **Interferometer Simetris dengan Pelat Setengah Gelombang (Gambar 2)**:
   Agar interferometer dapat bekerja untuk sumber cahaya dengan panjang koherensi pendek (seperti foton tunggal atau pulsa pendek), panjang kedua lintasan harus disamakan. Disisipkan pelat setengah gelombang ($\lambda/2$) yang sumbu cepatnya terorientasi pada sudut $45^\circ$ (matriks $\mathbf{J}_{\lambda/2, \theta=45^\circ} = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$).
   - Lintasan atas: Polarisator vertikal $\mathbf{J}_V$, pelat $\lambda/2$ memutar $V \to H$, lalu pergeseran fase $\mathbf{J}_\phi$.
   - Lintasan bawah: Polarisator horizontal $\mathbf{J}_H$, lalu pelat $\lambda/2$ memutar $H \to V$.
   
   Matriks Jones total untuk sistem interferometer internal pada Gambar 2 adalah:

   $$\mathbf{J}_{int} = \mathbf{J}_\phi \mathbf{J}_{\lambda/2, \theta=45^\circ} \mathbf{J}_V + \mathbf{J}_{\lambda/2, \theta=45^\circ} \mathbf{J}_H = \begin{pmatrix} e^{i\phi} & 0 \\ 0 & 1 \end{pmatrix} \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} \begin{pmatrix} 0 & 0 \\ 0 & 1 \end{pmatrix} + \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} \begin{pmatrix} 1 & 0 \\ 0 & 0 \end{pmatrix} = \begin{pmatrix} 0 & e^{i\phi} \\ 1 & 0 \end{pmatrix}. \tag{4}$$

4. **Interferometer Polarisasi Lengkap (Gambar 3)**:
   Untuk memodulasi intensitas berkas keluaran, ditambahkan polarisator $PA_{45}$ di bagian akhir sistem. Matriks Jones efektif kombinasi bersama saluran $+45^\circ$ dari $PA_{45}$ adalah:

   $$\mathbf{J} = \mathbf{J}_{+45} \mathbf{J}_{int} = \frac{1}{2} \begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix} \begin{pmatrix} 0 & e^{i\phi} \\ 1 & 0 \end{pmatrix} = \frac{1}{2} \begin{pmatrix} 1 & e^{i\phi} \\ 1 & e^{i\phi} \end{pmatrix}. \tag{5}$$

5. **Vektor Polarisasi & Intensitas Keluaran**:
   Jika berkas masukan terpolarisasi pada $+45^\circ$ ($\vec{P}_{in} = |+45\rangle$), vektor polarisasi berkas keluaran pada saluran $+45^\circ$ adalah:

   $$\vec{P} = \mathbf{J} \vec{P}_{+45} = \frac{1}{2} \begin{pmatrix} 1 & e^{i\phi} \\ 1 & e^{i\phi} \end{pmatrix} \frac{1}{\sqrt{2}} \begin{pmatrix} 1 \\ 1 \end{pmatrix} = \frac{1}{2\sqrt{2}} \begin{pmatrix} 1 + e^{i\phi} \\ 1 + e^{i\phi} \end{pmatrix} = \frac{1}{2}(1 + e^{i\phi}) \vec{P}_{45}. \tag{6}$$

   Intensitas keluarannya adalah intensitas masukan $I_i$ dikalikan dengan kuadrat magnitudo vektor tersebut:

   $$I = I_i \left| \frac{1}{2}(1 + e^{i\phi}) \right|^2 = \frac{I_i}{4}(1 + e^{i\phi})(1 + e^{-i\phi}) = \frac{I_i}{4}(2 + e^{i\phi} + e^{-i\phi}) = \frac{I_i}{2}(1 + \cos\phi). \tag{7}$$

   - Saat $\phi = 0$, $I = I_i$ (**interferensi konstruktif penuh** pada saluran $+45^\circ$).
   - Saat $\phi = \pi$, $I = 0$ (**interferensi destruktif penuh** pada saluran $+45^\circ$, seluruh energi keluar di saluran $-45^\circ$).

In [2]:
# Demonstrasi Numerik & Simbolik Bagian 1: Interferensi Gelombang Klasik

# 1. Verifikasi Matriks Pelat Setengah Gelombang pada theta = 45 derajat
HWP_45 = J_halfwave(45)
print_matrix("Matriks Pelat Setengah Gelombang (theta=45 deg)", HWP_45)

# 2. Verifikasi Penurunan Persamaan (4): J_int = J_phi @ HWP @ J_V + HWP @ J_H
phi_sym = sp.Symbol('phi', real=True)
J_phi_sym = sp.Matrix([[sp.exp(sp.I * phi_sym), 0], [0, 1]])
HWP_sym   = sp.Matrix([[0, 1], [1, 0]])
JV_sym    = sp.Matrix([[0, 0], [0, 1]])
JH_sym    = sp.Matrix([[1, 0], [0, 0]])

J_int_sym = sp.simplify(J_phi_sym * HWP_sym * JV_sym + HWP_sym * JH_sym)
print("Pembuktian Simbolik Persamaan (4) J_int:")
sp.pprint(J_int_sym)
print()

# 3. Verifikasi Penurunan Persamaan (5): J_total = J_+45 @ J_int
J_plus45_sym = (sp.S(1)/2) * sp.Matrix([[1, 1], [1, 1]])
J_total_sym = sp.simplify(J_plus45_sym * J_int_sym)
print("Pembuktian Simbolik Persamaan (5) J_total = J_+45 @ J_int:")
sp.pprint(J_total_sym)
print()

# 4. Verifikasi Vektor Keluaran Persamaan (6) dan Modulasi Intensitas Persamaan (7)
ket_plus45_sym = sp.Matrix([[1/sp.sqrt(2)], [1/sp.sqrt(2)]])
P_out_sym = sp.simplify(J_total_sym * ket_plus45_sym)
print("Vektor Keluaran P_out = J_total * |+45>:")
sp.pprint(P_out_sym)
print()

# Demonstrasi Numerik Modulasi Intensitas I = (I_i / 2)*(1 + cos(phi))
print("Verifikasi Numerik Modulasi Intensitas untuk I_i = 1.0:")
for phi_val in [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]:
    J_int_num = np.array([[0, np.exp(1j*phi_val)], [1, 0]], dtype=complex)
    ket_out_num = J_plus45 @ J_int_num @ ket_plus45
    I_num = inner_product(ket_out_num, ket_out_num).real
    I_analytic = 0.5 * (1 + np.cos(phi_val))
    print(f"  phi = {phi_val:6.4f} rad -> I_numerik = {I_num:.4f} | I_analitik = {I_analytic:.4f} | Cocok? {np.isclose(I_num, I_analytic)}")

Matriks Pelat Setengah Gelombang (theta=45 deg) =
[[0. 1.]
 [1. 0.]]

Pembuktian Simbolik Persamaan (4) J_int:
⎡    ⅈ⋅φ⎤
⎢0  ℯ   ⎥
⎢       ⎥
⎣1   0  ⎦

Pembuktian Simbolik Persamaan (5) J_total = J_+45 @ J_int:
⎡      ⅈ⋅φ⎤
⎢     ℯ   ⎥
⎢1/2  ────⎥
⎢      2  ⎥
⎢         ⎥
⎢      ⅈ⋅φ⎥
⎢     ℯ   ⎥
⎢1/2  ────⎥
⎣      2  ⎦

Vektor Keluaran P_out = J_total * |+45>:
⎡   ⎛ ⅈ⋅φ    ⎞⎤
⎢√2⋅⎝ℯ    + 1⎠⎥
⎢─────────────⎥
⎢      4      ⎥
⎢             ⎥
⎢   ⎛ ⅈ⋅φ    ⎞⎥
⎢√2⋅⎝ℯ    + 1⎠⎥
⎢─────────────⎥
⎣      4      ⎦

Verifikasi Numerik Modulasi Intensitas untuk I_i = 1.0:
  phi = 0.0000 rad -> I_numerik = 1.0000 | I_analitik = 1.0000 | Cocok? True
  phi = 1.5708 rad -> I_numerik = 0.5000 | I_analitik = 0.5000 | Cocok? True
  phi = 3.1416 rad -> I_numerik = 0.0000 | I_analitik = 0.0000 | Cocok? True
  phi = 4.7124 rad -> I_numerik = 0.5000 | I_analitik = 0.5000 | Cocok? True
  phi = 6.2832 rad -> I_numerik = 1.0000 | I_analitik = 1.0000 | Cocok? True


# 2. Interferensi Foton

## 2.1 Penjelasan Eksperimen (Gambar 4 & Gambar 5)

Ketika kita beralih dari berkas cahaya klasik ke level **foton tunggal** ($N$ foton dimasukkan satu per satu pada keadaan $|+45\rangle$), kita dapat menguji apakah foton menempuh satu lintasan tertentu atau berada dalam keadaan superposisi.

1. **Pemblokiran Lintasan Horizontal (Gambar 4a)**:
   Jika lintasan horizontal dari $PA_{HV}$ pertama diblokir, hanya foton yang melewati lintasan vertikal ($N/2$ foton) yang diteruskan. Pelat $\lambda/2$ merotasi $|V\rangle \to |H\rangle$. Setelah melewati $PA_{HV}$ kedua, berkas keluar seutuhnya pada saluran $|H\rangle$. Ketika memasuki $PA_{45}$ di akhir, probabilitas foton keluar di saluran $|+45\rangle$ adalah $50\%$ dan di saluran $|-45\rangle$ adalah $50\%$. Sehingga jumlah foton yang terdeteksi adalah:
   - Saluran $|+45\rangle$: $N/4$ foton ($25\%$ dari total masukan).
   - Saluran $|-45\rangle$: $N/4$ foton ($25\%$ dari total masukan).

2. **Pemblokiran Lintasan Vertikal (Gambar 4b)**:
   Jika sebaliknya lintasan vertikal diblokir, hanya lintasan horizontal ($N/2$ foton) yang diteruskan. Pelat $\lambda/2$ merotasi $|H\rangle \to |V\rangle$. Di $PA_{HV}$ kedua keluar di saluran $|V\rangle$, dan di $PA_{45}$ kembali terbagi rata:
   - Saluran $|+45\rangle$: $N/4$ foton ($25\%$ dari total masukan).
   - Saluran $|-45\rangle$: $N/4$ foton ($25\%$ dari total masukan).

3. **Tanpa Pemblokiran Lintasan (Gambar 4c / Gambar 5)**:
   Jika kedua lintasan terbuka (tanpa ada yang diblokir), intuisi klasik kita mungkin menduga hasil akhirnya adalah penjumlahan dari kedua kasus di atas ($N/4 + N/4 = N/2$ di $+45^\circ$ dan $N/2$ di $-45^\circ$).
   **Namun, eksperimen membuktikan hasil yang mengejutkan**:
   - Saluran $|+45\rangle$: **$N$ foton ($100\%$ dari total masukan)**.
   - Saluran $|-45\rangle$: **$0$ foton ($0\%$)**.

### Mengapa Hal Ini Terjadi?
Karena foton tidak diblokir, foton tunggal menempuh **kedua lintasan secara bersamaan (superposisi kuantum)**. Amplitudo probabilitas dari lintasan atas dan lintasan bawah berinterferensi secara **konstruktif** menuju saluran $|+45\rangle$ dan berinterferensi secara **destruktif** menuju saluran $|-45\rangle$ (untuk $\phi = 0$). Ketika kita memblokir salah satu jalur, kita merusak superposisi tersebut sehingga interferensi lenyap.

In [3]:
# Simulasi Kuantum Eksperimen Interferometer Foton (Gambar 4 & 5)

print("--- Simulasi Eksperimen Interferometer Foton (Input: |+45>) ---")
print()

# Kita tetapkan pergeseran fase phi = 0 untuk interferometer internal
J_int_0 = np.array([[0, 1], [1, 0]], dtype=complex) # Matriks internal saat phi=0

# Kasus 1: Gambar 4(a) - Lintasan H diblokir (hanya komponen V dari PA_HV pertama yang lewat)
psi_after_PA1_V = J_V @ ket_plus45 # Proyeksi ke V
psi_after_HWP_V = HWP_45 @ psi_after_PA1_V # Rotasi V -> H
psi_after_PA2_V = J_H @ psi_after_HWP_V # Keluar di saluran H dari PA_HV kedua

prob_plus45_caseA  = norm(J_plus45 @ psi_after_PA2_V)**2
prob_minus45_caseA = norm(J_minus45 @ psi_after_PA2_V)**2

print("1. Kasus Gambar 4(a) [Lintasan H diblokir]:")
print(f"   Probabilitas keluar di saluran |+45> = {prob_plus45_caseA*100:.1f}% (N/4 foton)")
print(f"   Probabilitas keluar di saluran |-45> = {prob_minus45_caseA*100:.1f}% (N/4 foton)")
print()

# Kasus 2: Gambar 4(b) - Lintasan V diblokir (hanya komponen H dari PA_HV pertama yang lewat)
psi_after_PA1_H = J_H @ ket_plus45 # Proyeksi ke H
psi_after_HWP_H = HWP_45 @ psi_after_PA1_H # Rotasi H -> V
psi_after_PA2_H = J_V @ psi_after_HWP_H # Keluar di saluran V dari PA_HV kedua

prob_plus45_caseB  = norm(J_plus45 @ psi_after_PA2_H)**2
prob_minus45_caseB = norm(J_minus45 @ psi_after_PA2_H)**2

print("2. Kasus Gambar 4(b) [Lintasan V diblokir]:")
print(f"   Probabilitas keluar di saluran |+45> = {prob_plus45_caseB*100:.1f}% (N/4 foton)")
print(f"   Probabilitas keluar di saluran |-45> = {prob_minus45_caseB*100:.1f}% (N/4 foton)")
print()

# Kasus 3: Gambar 4(c) / Gambar 5 - Tanpa pemblokiran (Kedua lintasan terbuka)
psi_out_unblocked = J_int_0 @ ket_plus45 # Superposisi utuh menempuh interferometer

prob_plus45_caseC  = norm(J_plus45 @ psi_out_unblocked)**2
prob_minus45_caseC = norm(J_minus45 @ psi_out_unblocked)**2

print("3. Kasus Gambar 4(c) / Gambar 5 [Tanpa Pemblokiran - Superposisi Utuh]:")
print(f"   Probabilitas keluar di saluran |+45> = {prob_plus45_caseC*100:.1f}% (N foton - Interferensi Konstruktif)")
print(f"   Probabilitas keluar di saluran |-45> = {prob_minus45_caseC*100:.1f}% (0 foton - Interferensi Destruktif)")
print()
print("Terbukti: Penjumlahan probabilitas klasik (25% + 25% = 50%) TIDAK SAMA dengan probabilitas kuantum (100%)!")

--- Simulasi Eksperimen Interferometer Foton (Input: |+45>) ---

1. Kasus Gambar 4(a) [Lintasan H diblokir]:
   Probabilitas keluar di saluran |+45> = 25.0% (N/4 foton)
   Probabilitas keluar di saluran |-45> = 25.0% (N/4 foton)

2. Kasus Gambar 4(b) [Lintasan V diblokir]:
   Probabilitas keluar di saluran |+45> = 25.0% (N/4 foton)
   Probabilitas keluar di saluran |-45> = 25.0% (N/4 foton)

3. Kasus Gambar 4(c) / Gambar 5 [Tanpa Pemblokiran - Superposisi Utuh]:
   Probabilitas keluar di saluran |+45> = 100.0% (N foton - Interferensi Konstruktif)
   Probabilitas keluar di saluran |-45> = 0.0% (0 foton - Interferensi Destruktif)

Terbukti: Penjumlahan probabilitas klasik (25% + 25% = 50%) TIDAK SAMA dengan probabilitas kuantum (100%)!


# 3. Latihan (Soal-Jawab)

## Soal 1
**Tunjukkan bahwa matriks Jones**
$$\mathbf{J} = \begin{pmatrix} 0 & e^{i\phi} \\ 1 & 0 \end{pmatrix}$$
**tidak memengaruhi amplitudo berkas cahaya. Dengan kata lain, buktikan bahwa untuk sembarang masukan vektor polarisasi satuan, keluaran dari operasi matriks Jones tersebut merupakan vektor satuan juga.**

### Pembuktian Analitik:
Misalkan kita memiliki sembarang vektor polarisasi satuan (vektor keadaan masukan) yang dinyatakan dalam basis horizontal-vertikal:
$$|\psi_{in}\rangle = \begin{pmatrix} c_H \\ c_V \end{pmatrix}.$$
Karena $|\psi_{in}\rangle$ adalah vektor satuan (ternormalisasi), berlaku syarat normalisasi:
$$\langle \psi_{in}|\psi_{in}\rangle = |c_H|^2 + |c_V|^2 = 1.$$

Terapkan operasi matriks Jones $\mathbf{J}$ pada vektor masukan tersebut untuk mendapatkan vektor keluaran $|\psi_{out}\rangle$:
$$|\psi_{out}\rangle = \mathbf{J}|\psi_{in}\rangle = \begin{pmatrix} 0 & e^{i\phi} \\ 1 & 0 \end{pmatrix} \begin{pmatrix} c_H \\ c_V \end{pmatrix} = \begin{pmatrix} e^{i\phi} c_V \\ c_H \end{pmatrix}.$$

Untuk membuktikan bahwa amplitudo berkas tidak terpengaruh, kita hitung norma atau panjang kuadrat dari vektor keluaran $|\psi_{out}\rangle$:
$$\langle \psi_{out}|\psi_{out}\rangle = |e^{i\phi} c_V|^2 + |c_H|^2 = |e^{i\phi}|^2 |c_V|^2 + |c_H|^2.$$
Karena besar dari eksponensial kompleks murni adalah satu ($|e^{i\phi}| = 1$), persamaan menjadi:
$$\langle \psi_{out}|\psi_{out}\rangle = (1)|c_V|^2 + |c_H|^2 = |c_V|^2 + |c_H|^2 = 1.$$

Terbukti bahwa $\langle \psi_{out}|\psi_{out}\rangle = 1$. Selain itu, sifat melestarikan norma ini ekuivalen dengan sifat **operator uniter** ($\mathbf{J}^\dagger \mathbf{J} = \mathbf{I}$):
$$\mathbf{J}^\dagger \mathbf{J} = \begin{pmatrix} 0 & 1 \\ e^{-i\phi} & 0 \end{pmatrix} \begin{pmatrix} 0 & e^{i\phi} \\ 1 & 0 \end{pmatrix} = \begin{pmatrix} 1 & 0 \\ 0 & e^{-i\phi}e^{i\phi} \end{pmatrix} = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix} = \mathbf{I}. \quad \blacksquare$$

In [4]:
# Verifikasi Komputasi Soal 1: Pelestarian Amplitudo & Sifat Uniter

print("Verifikasi Soal 1: Sifat Uniter dan Pelestarian Amplitudo Matriks J")
print()

# Verifikasi Uniteritas J^dagger @ J == I untuk berbagai sudut phi
for phi_val in [0.25, 1.37, np.pi, 4.88]:
    J_mat = np.array([[0, np.exp(1j*phi_val)], [1, 0]], dtype=complex)
    is_unitary = np.allclose(adjoint(J_mat) @ J_mat, np.eye(2))
    print(f"  phi = {phi_val:5.2f} rad -> Apakah J^dagger @ J == I? {is_unitary}")
print()

# Verifikasi pada 5 Vektor Keadaan Acak
np.random.seed(42)
for i in range(5):
    # Buat vektor acak kompleks dan normalisasi
    v_rnd = np.random.randn(2, 1) + 1j * np.random.randn(2, 1)
    v_rnd = v_rnd / norm(v_rnd)
    
    phi_rnd = np.random.uniform(0, 2*np.pi)
    J_rnd = np.array([[0, np.exp(1j*phi_rnd)], [1, 0]], dtype=complex)
    
    v_out = J_rnd @ v_rnd
    print(f"  Vektor acak #{i+1}: norma masukan = {norm(v_rnd):.6f} | norma keluaran = {norm(v_out):.6f} | Terjaga? {np.isclose(norm(v_rnd), norm(v_out))}")

Verifikasi Soal 1: Sifat Uniter dan Pelestarian Amplitudo Matriks J

  phi =  0.25 rad -> Apakah J^dagger @ J == I? True
  phi =  1.37 rad -> Apakah J^dagger @ J == I? True
  phi =  3.14 rad -> Apakah J^dagger @ J == I? True
  phi =  4.88 rad -> Apakah J^dagger @ J == I? True

  Vektor acak #1: norma masukan = 1.000000 | norma keluaran = 1.000000 | Terjaga? True
  Vektor acak #2: norma masukan = 1.000000 | norma keluaran = 1.000000 | Terjaga? True
  Vektor acak #3: norma masukan = 1.000000 | norma keluaran = 1.000000 | Terjaga? True
  Vektor acak #4: norma masukan = 1.000000 | norma keluaran = 1.000000 | Terjaga? True
  Vektor acak #5: norma masukan = 1.000000 | norma keluaran = 1.000000 | Terjaga? True


## Soal 2
**Lakukan analisis untuk intensitas saluran keluaran $-45^\circ$ dari sistem interferometer Gambar 3. Kombinasikan dengan hasil yang sudah diperoleh pada pers. (7), tunjukkan bahwa interferometer tersebut menjaga prinsip konservasi energi.**

### Pembuktian Analitik:
Sistem interferometer hingga persis sebelum masuk ke polarisator $PA_{45}$ memiliki matriks Jones:
$$\mathbf{J}_{int} = \begin{pmatrix} 0 & e^{i\phi} \\ 1 & 0 \end{pmatrix}.$$
Polarisator $PA_{45}$ membagi berkas menjadi saluran $+45^\circ$ dan $-45^\circ$. Untuk saluran $-45^\circ$, matriks Jones polarisator linier $-45^\circ$ adalah:
$$\mathbf{J}_{-45} = | -45\rangle\langle -45 | = \frac{1}{2} \begin{pmatrix} 1 & -1 \\ -1 & 1 \end{pmatrix}.$$

Matriks Jones efektif untuk keseluruhan sistem yang berujung pada saluran $-45^\circ$ adalah:
$$\mathbf{J}_{eff,-45} = \mathbf{J}_{-45} \mathbf{J}_{int} = \frac{1}{2} \begin{pmatrix} 1 & -1 \\ -1 & 1 \end{pmatrix} \begin{pmatrix} 0 & e^{i\phi} \\ 1 & 0 \end{pmatrix} = \frac{1}{2} \begin{pmatrix} -1 & e^{i\phi} \\ 1 & -e^{i\phi} \end{pmatrix}.$$

Untuk vektor masukan $|+45\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ 1 \end{pmatrix}$, vektor polarisasi keluaran pada saluran $-45^\circ$ adalah:
$$|-45\rangle_{out} = \mathbf{J}_{eff,-45} |+45\rangle = \frac{1}{2} \begin{pmatrix} -1 & e^{i\phi} \\ 1 & -e^{i\phi} \end{pmatrix} \frac{1}{\sqrt{2}} \begin{pmatrix} 1 \\ 1 \end{pmatrix} = \frac{1}{2\sqrt{2}} \begin{pmatrix} e^{i\phi} - 1 \\ -(e^{i\phi} - 1) \end{pmatrix} = \frac{1}{2}(e^{i\phi} - 1) \left( \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ -1 \end{pmatrix} \right) = \frac{1}{2}(e^{i\phi} - 1)|-45\rangle.$$

Intensitas pada saluran $-45^\circ$ ($I_{-45}$) adalah intensitas masukan $I_i$ dikalikan kuadrat magnitudo vektor keluaran tersebut:
$$I_{-45} = I_i \left| \frac{1}{2}(e^{i\phi} - 1) \right|^2 = \frac{I_i}{4} (e^{i\phi} - 1)(e^{-i\phi} - 1) = \frac{I_i}{4} (1 - e^{i\phi} - e^{-i\phi} + 1) = \frac{I_i}{4}(2 - 2\cos\phi) = \frac{I_i}{2}(1 - \cos\phi).$$

**Pembuktian Konservasi Energi**:
Kombinasikan dengan intensitas saluran $+45^\circ$ pada Persamaan (7), yaitu $I_{+45} = \frac{I_i}{2}(1 + \cos\phi)$:
$$I_{total} = I_{+45} + I_{-45} = \frac{I_i}{2}(1 + \cos\phi) + \frac{I_i}{2}(1 - \cos\phi) = \frac{I_i}{2}(1 + \cos\phi + 1 - \cos\phi) = \frac{I_i}{2}(2) = I_i.$$

Karena total intensitas keluaran ($I_{total}$) sama persis dengan intensitas masukan ($I_i$) untuk semua nilai fase $\phi$, terbukti bahwa interferometer menjaga prinsip konservasi energi. $\blacksquare$

In [5]:
# Verifikasi Komputasi Soal 2: Konservasi Energi I_+45 + I_-45 == I_i

print("Verifikasi Soal 2: Konservasi Energi pada Interferometer Lengkap")
print()

# Pembuktian Simbolik
phi_s = sp.Symbol('phi', real=True)
I_i_s = sp.Symbol('I_i', positive=True)

I_plus_s  = (I_i_s / 2) * (1 + sp.cos(phi_s))
I_minus_s = (I_i_s / 2) * (1 - sp.cos(phi_s))
I_tot_s   = sp.simplify(I_plus_s + I_minus_s)

print("Pembuktian Simbolik I_total = I_+45 + I_-45:")
print(f"  I_+45 = {I_plus_s}")
print(f"  I_-45 = {I_minus_s}")
print(f"  I_total = {I_tot_s}")
print()

# Verifikasi Numerik pada Grid Sudut phi
print("Verifikasi Numerik untuk I_i = 1.0:")
for phi_val in np.linspace(0, 2*np.pi, 7):
    J_int_num = np.array([[0, np.exp(1j*phi_val)], [1, 0]], dtype=complex)
    
    ket_out_plus  = J_plus45  @ J_int_num @ ket_plus45
    ket_out_minus = J_minus45 @ J_int_num @ ket_plus45
    
    I_plus_num  = inner_product(ket_out_plus, ket_out_plus).real
    I_minus_num = inner_product(ket_out_minus, ket_out_minus).real
    I_total_num = I_plus_num + I_minus_num
    
    print(f"  phi = {phi_val:5.2f} rad -> I_+45 = {I_plus_num:.4f} | I_-45 = {I_minus_num:.4f} | Total = {I_total_num:.4f} (Cocok? {np.isclose(I_total_num, 1.0)})")

Verifikasi Soal 2: Konservasi Energi pada Interferometer Lengkap

Pembuktian Simbolik I_total = I_+45 + I_-45:
  I_+45 = I_i*(cos(phi) + 1)/2
  I_-45 = I_i*(1 - cos(phi))/2
  I_total = I_i

Verifikasi Numerik untuk I_i = 1.0:
  phi =  0.00 rad -> I_+45 = 1.0000 | I_-45 = 0.0000 | Total = 1.0000 (Cocok? True)
  phi =  1.05 rad -> I_+45 = 0.7500 | I_-45 = 0.2500 | Total = 1.0000 (Cocok? True)
  phi =  2.09 rad -> I_+45 = 0.2500 | I_-45 = 0.7500 | Total = 1.0000 (Cocok? True)
  phi =  3.14 rad -> I_+45 = 0.0000 | I_-45 = 1.0000 | Total = 1.0000 (Cocok? True)
  phi =  4.19 rad -> I_+45 = 0.2500 | I_-45 = 0.7500 | Total = 1.0000 (Cocok? True)
  phi =  5.24 rad -> I_+45 = 0.7500 | I_-45 = 0.2500 | Total = 1.0000 (Cocok? True)
  phi =  6.28 rad -> I_+45 = 1.0000 | I_-45 = 0.0000 | Total = 1.0000 (Cocok? True)


## Soal 3
**Jika berkas foton yang memasuki sistem interferometer pada Gambar 5 bukanlah $|+45\rangle$, tetapi diganti menjadi $|V\rangle$, apakah kita akan mendapatkan interferensi? Jelaskan jika ya maupun jika tidak. Bagaimana lagi kalau berkas masukan diganti menjadi $|R\rangle$?**

### Jawaban Analitik:
Untuk menentukan apakah fenomena interferensi terjadi, prinsip utamanya adalah melihat apakah foton memasuki sistem dalam keadaan superposisi yang menempuh lebih dari satu lintasan dengan amplitudo probabilitas tidak nol di kedua lintasan tersebut.

#### 1. Kasus Masukan Berkas $|V\rangle$ (Tidak Terjadi Interferensi):
Jika foton masuk dalam keadaan terpolarisasi vertikal $|V\rangle$, saat bertemu pembagi berkas pertama $PA_{HV}$, foton $100\%$ ditransmisikan ke saluran vertikal (lintasan V) dan $0\%$ ke saluran horizontal (lintasan H).
Karena foton secara eksklusif hanya menempuh satu lintasan, tidak ada amplitudo probabilitas dari lintasan lain yang bisa berpadu saat mencapai penggabung $PA_{HV}$ kedua. Keluaran sistem hanyalah rotasi oleh pelat setengah gelombang pada lintasan V:
$$\mathbf{J}_{int}|V\rangle = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} \begin{pmatrix} 0 \\ 1 \end{pmatrix} = \begin{pmatrix} 1 \\ 0 \end{pmatrix} = |H\rangle.$$
Jika kita memblokir lintasan H di tengah, keluarannya tetap $|H\rangle$ ($100\%$). Tidak ada modulasi interferensi.

#### 2. Kasus Masukan Berkas $|R\rangle$ (Terjadi Interferensi Konstruktif):
Jika masukan diganti menjadi polarisasi melingkar kanan $|R\rangle = \frac{1}{\sqrt{2}}(|H\rangle - i|V\rangle) = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ -i \end{pmatrix}$, foton berada dalam superposisi dan terpecah ke kedua lintasan:
- Amplitudo melewati lintasan H: $\frac{1}{\sqrt{2}}$.
- Amplitudo melewati lintasan V: $-\frac{i}{\sqrt{2}}$.

Ketika kedua lintasan digabungkan kembali oleh $\mathbf{J}_{int}$ (untuk $\phi = 0$):
$$|\psi_{out}\rangle = \mathbf{J}_{int}|R\rangle = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ -i \end{pmatrix} = \frac{1}{\sqrt{2}}\begin{pmatrix} -i \\ 1 \end{pmatrix} = -i \left( \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ i \end{pmatrix} \right) = -i|L\rangle.$$

Probabilitas foton terukur dalam keadaan polarisasi melingkar kiri $|L\rangle$ di akhir sistem adalah $|-i|^2 = 100\%$.
Sebagai perbandingan, jika salah satu lintasan diblokir (misal hanya lintasan H yang dibuka), probabilitas mendapatkan $|L\rangle$ hanya $25\%$. Karena probabilitas total saat kedua lintasan terbuka ($100\%$) tidak sama dengan jumlah probabilitas masing-masing lintasan secara terpisah ($25\% + 25\% = 50\%$), terbukti terjadi interferensi kuantum!

In [6]:
# Verifikasi Komputasi Soal 3: Masukan |V> vs |R>

print("--- Verifikasi Soal 3: Analisis Interferensi Masukan |V> vs |R> ---")
print()

# 1. Kasus Masukan |V>
print("1. Kasus Masukan |V>:")
psi_out_V = J_int_0 @ ket_V
print_ket("Keluaran J_int |V>", psi_out_V)
print(f"   Probabilitas terukur sebagai |H> = {norm(outer_product(ket_H, ket_H) @ psi_out_V)**2 * 100:.1f}%")
# Jika lintasan H diblokir (di mana komponen H memang nol):
psi_blocked_V = J_H @ (HWP_45 @ (J_V @ ket_V)) # Hanya lewat V
print(f"   Probabilitas jika lintasan H diblokir = {norm(psi_blocked_V)**2 * 100:.1f}% (Sama persis -> TANPA INTERFERENSI)")
print()

# 2. Kasus Masukan |R>
print("2. Kasus Masukan |R>:")
psi_out_R = J_int_0 @ ket_R
print_ket("Keluaran J_int |R>", psi_out_R)

prob_L_unblocked = norm(outer_product(ket_L, ket_L) @ psi_out_R)**2
print(f"   Probabilitas terukur sebagai |L> (Tanpa Pemblokiran) = {prob_L_unblocked * 100:.1f}% (Interferensi Konstruktif Penuh)")

# Jika salah satu lintasan diblokir (misal lintasan V diblokir):
psi_only_H = J_V @ (HWP_45 @ (J_H @ ket_R)) # Hanya lewat H
prob_L_blocked = norm(outer_product(ket_L, ket_L) @ psi_only_H)**2
print(f"   Probabilitas terukur sebagai |L> (Jika 1 lintasan diblokir) = {prob_L_blocked * 100:.1f}%")
print()
print("Terbukti: Pada masukan |R>, 25% + 25% = 50% != 100% -> TERBUKTI TERJADI INTERFERENSI KUANTUM!")

--- Verifikasi Soal 3: Analisis Interferensi Masukan |V> vs |R> ---

1. Kasus Masukan |V>:
|Keluaran J_int |V>> =
[[1.]
 [0.]]

   Probabilitas terukur sebagai |H> = 100.0%
   Probabilitas jika lintasan H diblokir = 100.0% (Sama persis -> TANPA INTERFERENSI)

2. Kasus Masukan |R>:
|Keluaran J_int |R>> =
[[-0.    -0.7071j]
 [ 0.7071+0.j    ]]

   Probabilitas terukur sebagai |L> (Tanpa Pemblokiran) = 100.0% (Interferensi Konstruktif Penuh)
   Probabilitas terukur sebagai |L> (Jika 1 lintasan diblokir) = 25.0%

Terbukti: Pada masukan |R>, 25% + 25% = 50% != 100% -> TERBUKTI TERJADI INTERFERENSI KUANTUM!
